In [1]:
import sqlite3
import json
import os
from dotenv import load_dotenv
from supabase import create_client, Client

# 1. 환경 변수 및 클라이언트 설정
load_dotenv()
url = os.environ.get("SUPABASE_URL")
key = os.environ.get("SUPABASE_KEY")
supabase: Client = create_client(url, key)

def migrate_raw_ocr_data():
    # 2. SQLite에서 데이터 읽기
    conn = sqlite3.connect('database/ESGdata.db')
    cursor = conn.cursor()
    
    print("🚀 SQLite에서 raw_ocr_data 추출 중...")
    cursor.execute("SELECT file_name, raw_content, ocr_provider, processing_status FROM raw_ocr_data")
    rows = cursor.fetchall()
    
    ocr_records = []
    for row in rows:
        f_name, content_str, provider, status = row
        
        # [핵심] 문자열로 된 JSON을 파이썬 딕셔너리로 변환 (Supabase JSONB 입력용)
        try:
            content_json = json.loads(content_str)
        except:
            content_json = {"error": "Invalid JSON", "raw": content_str}

        ocr_records.append({
            "file_name": f_name,
            "raw_content": content_json,
            "ocr_provider": provider,
            "processing_status": "Pending", # 이관 시 모두 Pending으로 설정하여 재검증 유도
        })

    # 3. Supabase 벌크 적재
    if ocr_records:
        print(f"📦 총 {len(ocr_records)}건의 데이터를 Supabase로 전송합니다...")
        supabase.table("raw_ocr_data").insert(ocr_records).execute()
        print("✅ raw_ocr_data 이관 완료!")
    else:
        print("💡 이관할 데이터가 없습니다.")

    conn.close()

if __name__ == "__main__":
    migrate_raw_ocr_data()

🚀 SQLite에서 raw_ocr_data 추출 중...
📦 총 45건의 데이터를 Supabase로 전송합니다...
✅ raw_ocr_data 이관 완료!
